# Step 01b — Sessionize (DuckDB SQL)

Same **30-minute inactivity** rule as pandas step01, implemented in **SQL over Parquet**.

Companion script: `step01b_sessionize_duckdb_sql.py`

Writes:
- `day-1/output/sessionized_events_sql.parquet`
- `day-1/output/sessions_sql.parquet`


In [ ]:
from __future__ import annotations

from pathlib import Path
import duckdb

HERE = Path.cwd()
if (HERE / "day-1").exists():
    DAY1 = HERE / "day-1"
elif HERE.name == "01_sessionize":
    DAY1 = HERE.parent
else:
    DAY1 = HERE

DATA = DAY1 / "data"
OUTPUT = DAY1 / "output"
WEB_EVENTS_PARQUET = DATA / "web_events.parquet"
WEB_EVENTS_CSV = DATA / "web_events.csv"

SESSION_GAP_SECONDS = 30 * 60
DUPLICATE_GAP_SECONDS = 2
print("DAY1 =", DAY1)


## SQL


In [ ]:
SESSIONIZE_SQL = """
WITH raw AS (
    SELECT
        anonymous_id,
        CAST(timestamp AS TIMESTAMP) AS timestamp,
        event_name,
        content_id,
        device_type,
        referrer
    FROM read_parquet(?)
    WHERE anonymous_id IS NOT NULL
      AND timestamp IS NOT NULL
      AND event_name IS NOT NULL
),
ordered AS (
    SELECT
        *,
        lag(timestamp) OVER w AS prev_ts,
        lag(event_name) OVER w AS prev_event,
        lag(COALESCE(CAST(content_id AS VARCHAR), '')) OVER w AS prev_content
    FROM raw
    WINDOW w AS (PARTITION BY anonymous_id ORDER BY timestamp, event_name)
),
deduped AS (
    SELECT
        anonymous_id,
        timestamp,
        event_name,
        content_id,
        device_type,
        referrer
    FROM ordered
    WHERE NOT (
        prev_ts IS NOT NULL
        AND event_name = prev_event
        AND COALESCE(CAST(content_id AS VARCHAR), '') = prev_content
        AND date_diff('second', prev_ts, timestamp) <= ?
    )
),
with_gap AS (
    SELECT
        *,
        lag(timestamp) OVER w AS prev_event_ts,
        date_diff('second', lag(timestamp) OVER w, timestamp) AS gap_seconds
    FROM deduped
    WINDOW w AS (PARTITION BY anonymous_id ORDER BY timestamp, event_name)
),
flagged AS (
    SELECT
        *,
        CASE
            WHEN prev_event_ts IS NULL OR gap_seconds > ? THEN 1
            ELSE 0
        END AS new_session
    FROM with_gap
)
SELECT
    anonymous_id,
    timestamp,
    event_name,
    content_id,
    device_type,
    referrer,
    gap_seconds,
    SUM(new_session) OVER (
        ORDER BY anonymous_id, timestamp, event_name
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )::BIGINT AS session_idx,
    'S' || lpad(
        CAST(
            SUM(new_session) OVER (
                ORDER BY anonymous_id, timestamp, event_name
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS VARCHAR
        ),
        9,
        '0'
    ) AS session_id
FROM flagged
ORDER BY anonymous_id, timestamp, event_name
"""

SESSIONS_SQL = """
SELECT
    session_idx,
    ANY_VALUE(session_id) AS session_id,
    ANY_VALUE(anonymous_id) AS anonymous_id,
    MIN(timestamp) AS session_start,
    MAX(timestamp) AS session_end,
    arg_min(device_type, timestamp) AS device_type,
    arg_min(referrer, timestamp) AS referrer,
    COUNT(*)::BIGINT AS n_events,
    COUNT(DISTINCT content_id)::BIGINT AS n_content
FROM sessionized_events
GROUP BY session_idx
ORDER BY session_idx
"""


## Ensure parquet + run


In [ ]:
def ensure_web_events_parquet(con: duckdb.DuckDBPyConnection) -> Path:
    if WEB_EVENTS_PARQUET.exists():
        return WEB_EVENTS_PARQUET
    if not WEB_EVENTS_CSV.exists():
        raise FileNotFoundError(f"missing {WEB_EVENTS_PARQUET} and {WEB_EVENTS_CSV}")
    DATA.mkdir(parents=True, exist_ok=True)
    csv_sql = WEB_EVENTS_CSV.as_posix().replace("'", "''")
    parquet_sql = WEB_EVENTS_PARQUET.as_posix().replace("'", "''")
    con.execute(
        f"""
        COPY (
            SELECT * FROM read_csv_auto('{csv_sql}', header := true, sample_size := -1)
        ) TO '{parquet_sql}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )
    print(f"wrote {WEB_EVENTS_PARQUET}")
    return WEB_EVENTS_PARQUET


OUTPUT.mkdir(parents=True, exist_ok=True)
events_out = OUTPUT / "sessionized_events_sql.parquet"
sessions_out = OUTPUT / "sessions_sql.parquet"

con = duckdb.connect()
try:
    parquet_path = ensure_web_events_parquet(con)
    con.execute(
        "CREATE OR REPLACE TEMP TABLE sessionized_events AS " + SESSIONIZE_SQL,
        [str(parquet_path), DUPLICATE_GAP_SECONDS, SESSION_GAP_SECONDS],
    )
    con.execute("CREATE OR REPLACE TEMP TABLE sessions AS " + SESSIONS_SQL)
    con.execute(
        f"COPY sessionized_events TO '{events_out.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    con.execute(
        f"COPY sessions TO '{sessions_out.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    n_events, n_sessions, n_cookies = con.execute(
        """
        SELECT
            (SELECT count(*) FROM sessionized_events),
            (SELECT count(*) FROM sessions),
            (SELECT count(DISTINCT anonymous_id) FROM sessions)
        """
    ).fetchone()
    print("DuckDB sessionization over parquet")
    print(f"  source:  {parquet_path}")
    print(f"  events:  {n_events:,}")
    print(f"  sessions:{n_sessions:,}")
    print(f"  cookies: {n_cookies:,}")
    print(f"  wrote {events_out}")
    print(f"  wrote {sessions_out}")
finally:
    con.close()


## Query outputs


In [ ]:
con = duckdb.connect()
print(con.execute(f"SELECT count(*) AS n_events FROM read_parquet('{events_out.as_posix()}')").fetchdf())
print(con.execute(f"SELECT count(*) AS n_sessions FROM read_parquet('{sessions_out.as_posix()}')").fetchdf())
display(
    con.execute(
        f"""
        SELECT session_id, anonymous_id, session_start, session_end, n_events, n_content
        FROM read_parquet('{sessions_out.as_posix()}')
        ORDER BY session_start
        LIMIT 10
        """
    ).fetchdf()
)
con.close()
